In [1]:
import numpy as np
from typing import List, Dict

class CircleGeometryGenerator:
    """
    Generates parameterized circle blueprints for synthetic level-set data.
    Strictly follows the mathematical bounds and sampling rules defined in the paper.
    """
    def __init__(self, resolution_rho: int, seed: int = 42, variations: int = 5):
        """
        Generate circle blueprints for development-stage synthetic data.

        Parameters
        ----------
        resolution_rho : int
            Grid resolution (paper uses 256, 266, 276).
        seed : int
            Global seed controlling deterministic blueprint generation.
        variations : int
            Number of center jitters per radius (paper says up to 5).
        """
        self.rho = int(resolution_rho)
        self.global_seed = int(seed)
        self.variations = int(variations)

        # Optional strict reproduction checks
        if self.rho not in (256, 266, 276):
            print(
                f"[Warning] rho={self.rho} is not one of the paper's listed "
                f"training resolutions (256, 266, 276)."
            )
        if self.variations < 1:
            raise ValueError("variations must be >= 1")
        if self.variations > 5:
            print(
                f"[Warning] variations={self.variations} exceeds the paper's "
                f"'up to 5' setting."
            )

        # Grid spacing definition: h = 1 / (rho - 1)
        self.h = 1.0 / (self.rho - 1)
        
        # Radius bounds from paper logic
        self.r_min = 1.6 * self.h
        self.r_max = 0.5 - 2.0 * self.h
        
        # Formula for the number of distinct radii
        self.num_radii = int(np.floor((self.rho - 8.2) / 2.0)) + 1
        
        # Defensive check: Ensure resolution is large enough to form valid circles
        if self.num_radii < 1 or self.r_min >= self.r_max:
            raise ValueError(
                f"[Error] Resolution rho={self.rho} is too small to generate valid "
                f"circular interfaces. Required: num_radii >= 1 and r_min < r_max."
            )

        self.radii_set = np.linspace(self.r_min, self.r_max, self.num_radii, dtype=float)

        if not np.isclose(self.radii_set[0], self.r_min, atol=1e-12, rtol=0.0):
            raise RuntimeError(
                f"First radius mismatch: {self.radii_set[0]} vs r_min={self.r_min}"
            )
        if not np.isclose(self.radii_set[-1], self.r_max, atol=1e-12, rtol=0.0):
            raise RuntimeError(
                f"Last radius mismatch: {self.radii_set[-1]} vs r_max={self.r_max}"
            )

        self.center_min = 0.5 - self.h / 2.0
        self.center_max = 0.5 + self.h / 2.0

    def _subseed(self, r_idx: int, v_idx: int) -> int:
        """
        Generates a deterministic per-blueprint pseudo-random seed.
        Note: This is used ONLY for RNG control, NOT as a unique global ID.
        """
        x = np.uint64(self.global_seed)
        x ^= np.uint64(1469598103934665603)
        x ^= np.uint64(r_idx + 1) * np.uint64(1099511628211)
        x ^= np.uint64(v_idx + 1) * np.uint64(14029467366897019727)
        return int(x % np.uint64(2**32))

    def generate_blueprints(self) -> List[Dict]:
        blueprints = []
        for r_idx, r in enumerate(self.radii_set):
            analytic_kappa = 1.0 / r
            analytic_h_kappa = self.h / r

            for v_idx in range(self.variations):
                sub_seed = self._subseed(r_idx, v_idx)
                local_rng = np.random.default_rng(sub_seed)
                
                cx = local_rng.uniform(self.center_min, self.center_max)
                cy = local_rng.uniform(self.center_min, self.center_max)

                blueprint_id_str = f"rho{self.rho}_r{r_idx:03d}_v{v_idx:02d}_s{sub_seed}"
                
                blueprint_idx_int = int(self.rho * 100000 + r_idx * self.variations + v_idx)

                blueprints.append({
                    "meta": {
                        "blueprint_id": blueprint_id_str,
                        "blueprint_idx": blueprint_idx_int,  
                        "geometry_type": "circle",
                        "resolution": self.rho,
                        "radius_idx": int(r_idx),           
                        "variation_idx": int(v_idx),
                        "global_seed": self.global_seed,
                        "sub_seed": sub_seed,
                    },
                    "params": {
                        "h": float(self.h), 
                        "radius": float(r), 
                        "center": (float(cx), float(cy))
                    },
                    "label": {
                        "source": "analytic_circle", 
                        "kappa": float(analytic_kappa), 
                        "h_kappa": float(analytic_h_kappa)  
                    }
                })
        return blueprints


if __name__ == "__main__":
    rho_test = 256

    gen1 = CircleGeometryGenerator(rho_test, seed=42, variations=5)
    bps1 = gen1.generate_blueprints()

    gen2 = CircleGeometryGenerator(rho_test, seed=42, variations=5)
    bps2 = gen2.generate_blueprints()

    print(f"Resolution: {rho_test}")
    print(f"h = {gen1.h:.12f}")
    print(f"Radius range = [{gen1.r_min:.12f}, {gen1.r_max:.12f}]")
    print(f"Num Radii: {gen1.num_radii}")
    print(f"Total Blueprints: {len(bps1)}")

    # Reproducibility checks
    assert len(bps1) == gen1.num_radii * gen1.variations
    assert bps1[0]["params"] == bps2[0]["params"], "Reproducibility failed on first blueprint."
    assert bps1[-1]["params"] == bps2[-1]["params"], "Reproducibility failed on last blueprint."

    # Stronger array-level reproducibility check
    arr1 = np.array(
        [[bp["params"]["radius"], *bp["params"]["center"]] for bp in bps1],
        dtype=float
    )
    arr2 = np.array(
        [[bp["params"]["radius"], *bp["params"]["center"]] for bp in bps2],
        dtype=float
    )
    assert np.allclose(arr1, arr2), "Full blueprint array reproducibility failed."

    # Radius endpoint coverage and monotonicity checks (linspace version)
    assert np.isclose(gen1.radii_set[0], gen1.r_min, atol=1e-12, rtol=0.0)
    assert np.isclose(gen1.radii_set[-1], gen1.r_max, atol=1e-12, rtol=0.0)
    dr = np.diff(gen1.radii_set)
    assert np.all(dr > 0), "Radii must be strictly increasing."

    # Range checks for centers
    for bp in bps1[:10]:
        cx, cy = bp["params"]["center"]
        assert gen1.center_min <= cx <= gen1.center_max
        assert gen1.center_min <= cy <= gen1.center_max

    # Label monotonicity (pick one variation per radius)
    labels = np.array([bps1[i * gen1.variations]["label"]["h_kappa"] for i in range(gen1.num_radii)])
    assert np.all(np.diff(labels) < 0), "h/r labels should decrease as radius increases."

    # Same radius -> same labels across variations
    for r_idx in range(gen1.num_radii):
        vals = [bps1[r_idx * gen1.variations + v]["label"]["h_kappa"] for v in range(gen1.variations)]
        assert np.allclose(vals, vals[0]), f"Label mismatch at radius_idx={r_idx}"

    print(f"Sample first blueprint = {bps1[0]}")
    print("All checks passed.")

Resolution: 256
h = 0.003921568627
Radius range = [0.006274509804, 0.492156862745]
Num Radii: 124
Total Blueprints: 620
Sample first blueprint = {'meta': {'blueprint_id': 'rho256_r000_v00_s1414130005', 'blueprint_idx': 25600000, 'geometry_type': 'circle', 'resolution': 256, 'radius_idx': 0, 'variation_idx': 0, 'global_seed': 42, 'sub_seed': 1414130005}, 'params': {'h': 0.00392156862745098, 'radius': 0.006274509803921569, 'center': (0.49887091758458463, 0.501719777278703)}, 'label': {'source': 'analytic_circle', 'kappa': 159.37499999999997, 'h_kappa': 0.6249999999999999}}
All checks passed.


C:\Users\29801\AppData\Local\Temp\ipykernel_74320\477372973.py:79: RuntimeWarning: overflow encountered in scalar multiply
  x ^= np.uint64(v_idx + 1) * np.uint64(14029467366897019727)


In [2]:
import numpy as np
from typing import Dict, Any


class LevelSetFieldBuilder:
    """
    Build level-set fields from geometry blueprints.

    Indexing convention (IMPORTANT):
        phi[i, j] <-> point (x_i, y_j)
    achieved by np.meshgrid(..., indexing='ij')
    """

    def __init__(self, dtype=np.float64):
        self.dtype = dtype

    def _build_grid(self, rho: int):
        """
        Build uniform grid on [0,1]x[0,1] with indexing='ij'.
        Returns
        -------
        x : (rho,) ndarray
            x_i = i*h
        y : (rho,) ndarray
            y_j = j*h
        X : (rho, rho) ndarray
            X[i, j] = x_i
        Y : (rho, rho) ndarray
            Y[i, j] = y_j
        h : float
            grid spacing
        """
        rho = int(rho)
        if rho < 2:
            raise ValueError("rho must be >= 2")

        h = 1.0 / (rho - 1)
        x = np.linspace(0.0, 1.0, rho, dtype=self.dtype)
        y = np.linspace(0.0, 1.0, rho, dtype=self.dtype)

        X, Y = np.meshgrid(x, y, indexing='ij')
        return x, y, X, Y, float(h)

    def _parse_blueprint(self, blueprint: Dict[str, Any]):
        """
        Parse and validate minimal required fields from blueprint.
        """
        if "label" not in blueprint:
            raise KeyError("Blueprint missing 'label'; downstream dataset builder may fail.")

        try:
            rho = int(blueprint["meta"]["resolution"])
            h_bp = float(blueprint["params"]["h"])
            r = float(blueprint["params"]["radius"])
            cx, cy = blueprint["params"]["center"]
            cx = float(cx)
            cy = float(cy)
        except KeyError as e:
            raise KeyError(f"Blueprint missing required key: {e}")

        return rho, h_bp, r, cx, cy

    def _pack_output(
        self,
        blueprint: Dict[str, Any],
        phi: np.ndarray,
        phi_type: str,
        return_grid: bool,
        x: np.ndarray,
        y: np.ndarray,
        X: np.ndarray,
        Y: np.ndarray,
        h: float,
    ) -> Dict[str, Any]:
        """
        Standardize output structure and pass through label (IMPORTANT).
        """
        out = {
            "meta": {
                **blueprint["meta"],
                "stage": "level_set_field",
            },
            "params": {
                **blueprint["params"],
            },
            "label": {
                **blueprint["label"],  
            },
            "field": {
                "phi_type": phi_type,
                "indexing": "ij",
                "phi": phi.astype(self.dtype, copy=False),
            },
        }

        if return_grid:
            out["grid"] = {
                "x": x,
                "y": y,
                "X": X,
                "Y": Y,
                "h": float(h),
            }

        return out

    def build_circle_sdf(self, blueprint: Dict[str, Any], return_grid: bool = True) -> Dict[str, Any]:
        """
        Build signed-distance level-set field for a circle:

            phi_cs(x,y) = sqrt((x-cx)^2 + (y-cy)^2) - r
        """
        rho, h_bp, r, cx, cy = self._parse_blueprint(blueprint)
        x, y, X, Y, h = self._build_grid(rho)

        if not np.isclose(h, h_bp, atol=1e-15, rtol=0.0):
            raise ValueError(f"Grid spacing mismatch: blueprint h={h_bp}, rebuilt h={h}")

        phi = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2) - r

        return self._pack_output(
            blueprint=blueprint,
            phi=phi,
            phi_type="circle_sdf",
            return_grid=return_grid,
            x=x, y=y, X=X, Y=Y, h=h
        )

    def build_circle_nonsdf(self, blueprint: Dict[str, Any], return_grid: bool = True) -> Dict[str, Any]:
        """
        Build non-signed-distance circle level-set field:

            phi_cn(x,y) = (x-cx)^2 + (y-cy)^2 - r^2

        Note:
            This is NOT a signed-distance function and is intended as input
            to the reinitialization stage.
        """
        rho, h_bp, r, cx, cy = self._parse_blueprint(blueprint)
        x, y, X, Y, h = self._build_grid(rho)

        if not np.isclose(h, h_bp, atol=1e-15, rtol=0.0):
            raise ValueError(f"Grid spacing mismatch: blueprint h={h_bp}, rebuilt h={h}")

        phi = (X - cx) ** 2 + (Y - cy) ** 2 - r ** 2

        return self._pack_output(
            blueprint=blueprint,
            phi=phi,
            phi_type="circle_nonsdf",
            return_grid=return_grid,
            x=x, y=y, X=X, Y=Y, h=h
        )

    @staticmethod
    def quick_sanity_checks(field_pack: Dict[str, Any], verbose: bool = True) -> None:
        """
        Quick anti-bug checks for circle fields (sdf or nonsdf).

        Checks:
        1) A grid point nearest to the center should be inside the circle (phi < 0).
        2) A corner point should be outside (typically phi > 0).
        3) There should exist sign changes on grid edges (interface crosses some edges).
        4) label is present and includes h_kappa (for downstream use).
        """
        if "label" not in field_pack:
            raise KeyError("field_pack missing 'label' (dataflow bug).")
        if "h_kappa" not in field_pack["label"]:
            raise KeyError("field_pack['label'] missing 'h_kappa'.")

        phi = field_pack["field"]["phi"]
        phi_type = field_pack["field"]["phi_type"]
        x = field_pack["grid"]["x"]
        y = field_pack["grid"]["y"]
        h = field_pack["grid"]["h"]

        cx, cy = field_pack["params"]["center"]
        r = field_pack["params"]["radius"]
        h_kappa = field_pack["label"]["h_kappa"]

        rho = phi.shape[0]
        assert phi.shape == (rho, rho), "phi must be square (rho, rho)"

        i0 = int(np.argmin(np.abs(x - cx)))
        j0 = int(np.argmin(np.abs(y - cy)))
        phi_center_near = float(phi[i0, j0])
        assert phi_center_near < 0.0, (
            f"Nearest grid point to center is not inside circle: phi[{i0},{j0}]={phi_center_near}"
        )

        phi_corner = float(phi[0, 0])
        assert phi_corner > 0.0, f"Corner point unexpectedly not outside: phi[0,0]={phi_corner}"

        sign_change_i = np.any(phi[:-1, :] * phi[1:, :] <= 0.0)
        sign_change_j = np.any(phi[:, :-1] * phi[:, 1:] <= 0.0)
        assert (sign_change_i or sign_change_j), "No sign-changing edges found; interface may be missing."

        phi_min = float(np.min(phi))
        phi_max = float(np.max(phi))
        min_abs_phi = float(np.min(np.abs(phi)))

        if verbose:
            print("[Sanity Checks Passed]")
            print(f"  phi_type             : {phi_type}")
            print(f"  indexing convention  : phi[i,j] <-> (x_i, y_j)")
            print(f"  rho                  : {rho}")
            print(f"  h                    : {h:.12f}")
            print(f"  center               : ({cx:.12f}, {cy:.12f})")
            print(f"  radius               : {r:.12f}")
            print(f"  label h_kappa        : {h_kappa:.12f}")
            print(f"  nearest-center idx   : (i0,j0)=({i0},{j0})")
            print(f"  phi[i0,j0]           : {phi_center_near:.12f}  (should be < 0)")
            print(f"  phi[0,0]             : {phi_corner:.12f}  (should be > 0)")
            print(f"  sign-change edges    : i-dir={bool(sign_change_i)}, j-dir={bool(sign_change_j)}")
            print(f"  min(phi), max(phi)   : ({phi_min:.12f}, {phi_max:.12f})")
            print(f"  min |phi| on grid    : {min_abs_phi:.12e} (not necessarily 0 due to grid alignment)")


if __name__ == "__main__":
    class CircleGeometryGenerator:
        def __init__(self, resolution_rho: int, seed: int = 42, variations: int = 5):
            self.rho = int(resolution_rho)
            self.global_seed = int(seed)
            self.variations = int(variations)

            self.h = 1.0 / (self.rho - 1)
            self.r_min = 1.6 * self.h
            self.r_max = 0.5 - 2.0 * self.h
            self.num_radii = int(np.floor((self.rho - 8.2) / 2.0)) + 1
            self.radii_set = np.linspace(self.r_min, self.r_max, self.num_radii, dtype=float)

            self.center_min = 0.5 - self.h / 2.0
            self.center_max = 0.5 + self.h / 2.0

        def _subseed(self, r_idx: int, v_idx: int) -> int:
            x = np.uint64(self.global_seed)
            x ^= np.uint64(1469598103934665603)
            x ^= np.uint64(r_idx + 1) * np.uint64(1099511628211)
            x ^= np.uint64(v_idx + 1) * np.uint64(14029467366897019727)
            return int(x % np.uint64(2**32))

        def generate_blueprints(self):
            blueprints = []
            for r_idx, r in enumerate(self.radii_set):
                for v_idx in range(self.variations):
                    sub_seed = self._subseed(r_idx, v_idx)
                    rng = np.random.default_rng(sub_seed)
                    cx = rng.uniform(self.center_min, self.center_max)
                    cy = rng.uniform(self.center_min, self.center_max)
                    blueprints.append({
                        "meta": {
                            "blueprint_id": f"rho{self.rho}_r{r_idx:03d}_v{v_idx:02d}_s{sub_seed}",
                            "geometry_type": "circle",
                            "resolution": self.rho,
                            "radius_idx": r_idx,
                            "variation_idx": v_idx,
                            "global_seed": self.global_seed,
                            "sub_seed": sub_seed,
                        },
                        "params": {
                            "h": float(self.h),
                            "radius": float(r),
                            "center": (float(cx), float(cy)),
                        },
                        "label": {
                            "source": "analytic_circle",
                            "kappa": float(1.0 / r),
                            "h_kappa": float(self.h / r),
                        }
                    })
            return blueprints

    gen = CircleGeometryGenerator(resolution_rho=256, seed=42, variations=5)
    blueprints = gen.generate_blueprints()
    bp = blueprints[0]

    builder = LevelSetFieldBuilder(dtype=np.float64)

    # SDF field
    pack_sdf = builder.build_circle_sdf(bp, return_grid=True)
    print(f"[SDF] blueprint_id = {pack_sdf['meta']['blueprint_id']}")
    print(f"[SDF] phi shape    = {pack_sdf['field']['phi'].shape}")
    builder.quick_sanity_checks(pack_sdf, verbose=True)
    print()

    # Non-SDF field (for next-stage reinitialization)
    pack_nonsdf = builder.build_circle_nonsdf(bp, return_grid=True)
    print(f"[NonSDF] blueprint_id = {pack_nonsdf['meta']['blueprint_id']}")
    print(f"[NonSDF] phi shape     = {pack_nonsdf['field']['phi'].shape}")
    builder.quick_sanity_checks(pack_nonsdf, verbose=True)

[SDF] blueprint_id = rho256_r000_v00_s1414130005
[SDF] phi shape    = (256, 256)
[Sanity Checks Passed]
  phi_type             : circle_sdf
  indexing convention  : phi[i,j] <-> (x_i, y_j)
  rho                  : 256
  h                    : 0.003921568627
  center               : (0.498870917585, 0.501719777279)
  radius               : 0.006274509804
  label h_kappa        : 0.625000000000
  nearest-center idx   : (i0,j0)=(127,128)
  phi[i0,j0]           : -0.005408592696  (should be < 0)
  phi[0,0]             : 0.701252823466  (should be > 0)
  sign-change edges    : i-dir=True, j-dir=True
  min(phi), max(phi)   : (-0.005408592696, 0.702846842403)
  min |phi| on grid    : 4.376660953847e-05 (not necessarily 0 due to grid alignment)

[NonSDF] blueprint_id = rho256_r000_v00_s1414130005
[NonSDF] phi shape     = (256, 256)
[Sanity Checks Passed]
  phi_type             : circle_nonsdf
  indexing convention  : phi[i,j] <-> (x_i, y_j)
  rho                  : 256
  h                    :

C:\Users\29801\AppData\Local\Temp\ipykernel_74320\3819931530.py:239: RuntimeWarning: overflow encountered in scalar multiply
  x ^= np.uint64(v_idx + 1) * np.uint64(14029467366897019727)


In [3]:
import copy
import numpy as np
from typing import Dict, Any, List, Tuple

class ReinitQualityEvaluator:
    
    """
    Evaluates the quality of a level-set field (|∇φ| ≈ 1) 
    strictly on the grid nodes to be sampled for training.
    """

    @staticmethod
    def _central_grad_norm(phi: np.ndarray, h: float) -> np.ndarray:
        """Computes gradient norm using central differences for diagnostics."""
        dy, dx = np.gradient(phi, h, h, edge_order=1)
        return np.sqrt(dx**2 + dy**2)

    @staticmethod
    def get_sampling_coordinates(phi: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        [PUBLIC API] 
        Finds the (i, j) indices of grid nodes whose OUTGOING vertical
        or horizontal edges are crossed by the interface.
        
        Note: If a node has both its outgoing x-edge and y-edge crossed, 
        its coordinates will appear twice. This duplicates the sampling logic
        from the reference paper exactly.
        """
        # Sign changes across OUTGOING edges
        sign_change_x = phi[:-1, :] * phi[1:, :] <= 0.0
        sign_change_y = phi[:, :-1] * phi[:, 1:] <= 0.0
        
        # Zero out the outermost boundary to prevent 3x3 stencil out-of-bounds later
        sign_change_x[0, :] = sign_change_x[-1, :] = False
        sign_change_x[:, 0] = sign_change_x[:, -1] = False
        
        sign_change_y[0, :] = sign_change_y[-1, :] = False
        sign_change_y[:, 0] = sign_change_y[:, -1] = False
        
        # Extract coordinates independently
        Ix, Jx = np.where(sign_change_x)
        Iy, Jy = np.where(sign_change_y)
        
        # Concatenate to allow identical stencils (matches the paper's 3.14M count)
        I = np.concatenate([Ix, Iy])
        J = np.concatenate([Jx, Jy])
        
        return I, J

    @classmethod
    def evaluate(cls, phi: np.ndarray, h: float) -> Dict[str, float]:
        """Compute SDF quality metrics exactly on the nodes to be sampled."""
        grad_norm = cls._central_grad_norm(phi, h)
        
        I, J = cls.get_sampling_coordinates(phi)

        if len(I) == 0:
            return {
                "mean_abs_err_to_1": float('nan'),
                "max_abs_err_to_1": float('nan'),
                "grad_norm_std": float('nan'),
            }

        # Evaluate only at the selected coordinates (duplicates don't skew the max/mean)
        grad_in_band = grad_norm[I, J]
        abs_err = np.abs(grad_in_band - 1.0)

        return {
            "mean_abs_err_to_1": float(np.mean(abs_err)),
            "max_abs_err_to_1": float(np.max(abs_err)),
            "grad_norm_std": float(np.std(grad_in_band)),
        }
class LevelSetReinitializer:
    """
    Level-set reinitialization PDE solver implementing:
    - 5th-order Hamilton-Jacobi WENO (Jiang & Peng, 2000)
    - Classic Rouy-Tourin Godunov upwind Hamiltonian
    - 3rd-order TVD Runge-Kutta (SSP-RK3)
    """
    def __init__(self, cfl: float = 0.5, eps_weno: float = 1e-6, eps_sign_factor: float = 1.0):
        self.cfl = cfl
        self.eps_weno = eps_weno
        self.eps_sign_factor = eps_sign_factor

    def _smoothed_sign(self, phi0: np.ndarray, h: float) -> np.ndarray:
        """Smoothed sign function S(phi0) = phi0 / sqrt(phi0^2 + eps^2)."""
        eps = self.eps_sign_factor * h
        return phi0 / np.sqrt(phi0**2 + eps**2)

    def _hj_weno5_1d(self, v1: np.ndarray, v2: np.ndarray, v3: np.ndarray, v4: np.ndarray, v5: np.ndarray) -> np.ndarray:
        """
        Inputs v1, v2, v3, v4, v5 represent the 1st-order difference quotients 
        (delta_{i-5/2}, ..., delta_{i+3/2}) for D^- 
        or their flipped counterparts for D^+.
        """
        # 1. Smoothness Indicators (beta_k)
        beta0 = (13.0/12.0) * (v1 - 2.0*v2 + v3)**2 + (1.0/4.0) * (v1 - 4.0*v2 + 3.0*v3)**2
        beta1 = (13.0/12.0) * (v2 - 2.0*v3 + v4)**2 + (1.0/4.0) * (v2 - v4)**2
        beta2 = (13.0/12.0) * (v3 - 2.0*v4 + v5)**2 + (1.0/4.0) * (3.0*v3 - 4.0*v4 + v5)**2

        # 2. Linear and Nonlinear Weights (alpha_k and omega_k)
        alpha0 = 0.1 / (beta0 + self.eps_weno)**2
        alpha1 = 0.6 / (beta1 + self.eps_weno)**2
        alpha2 = 0.3 / (beta2 + self.eps_weno)**2
        sum_alpha = alpha0 + alpha1 + alpha2

        omega0 = alpha0 / sum_alpha
        omega1 = alpha1 / sum_alpha
        omega2 = alpha2 / sum_alpha

        # 3. Polynomial Approximations (p_k)
        p0 = (1.0/3.0)*v1 - (7.0/6.0)*v2 + (11.0/6.0)*v3
        p1 = -(1.0/6.0)*v2 + (5.0/6.0)*v3 + (1.0/3.0)*v4
        p2 = (1.0/3.0)*v3 + (5.0/6.0)*v4 - (1.0/6.0)*v5

        # 4. Final WENO5 Approximation
        return omega0*p0 + omega1*p1 + omega2*p2

    def _get_derivatives_weno5(self, phi: np.ndarray, h: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Computes the four one-sided spatial derivatives (Dx_m, Dx_p, Dy_m, Dy_p)
        using the standard HJ-WENO5 difference quotient mapping.
        """
        nx, ny = phi.shape
        
        # Pad domain with constant extrapolation (3 layers needed for 5-point stencil)
        phi_pad = np.pad(phi, pad_width=3, mode='edge')

        # Forward difference quotients: d_x[i] corresponds to delta_{i+1/2}
        d_x = (phi_pad[1:, :] - phi_pad[:-1, :]) / h
        d_y = (phi_pad[:, 1:] - phi_pad[:, :-1]) / h

        # ---------------------------------------------------------------------
        # X-Direction Derivatives
        # ---------------------------------------------------------------------
        # Left-biased derivative (D^-_x)
        Dx_m = self._hj_weno5_1d(
            v1=d_x[0:nx, 3:-3],   # delta_{i-5/2}
            v2=d_x[1:nx+1, 3:-3], # delta_{i-3/2}
            v3=d_x[2:nx+2, 3:-3], # delta_{i-1/2}
            v4=d_x[3:nx+3, 3:-3], # delta_{i+1/2}
            v5=d_x[4:nx+4, 3:-3]  # delta_{i+3/2}
        )
        
        # Right-biased derivative (D^+_x) using flipped inputs (v_tilde)
        Dx_p = self._hj_weno5_1d(
            v1=d_x[5:nx+5, 3:-3], # delta_{i+5/2}
            v2=d_x[4:nx+4, 3:-3], # delta_{i+3/2}
            v3=d_x[3:nx+3, 3:-3], # delta_{i+1/2}
            v4=d_x[2:nx+2, 3:-3], # delta_{i-1/2}
            v5=d_x[1:nx+1, 3:-3]  # delta_{i-3/2}
        )

        # ---------------------------------------------------------------------
        # Y-Direction Derivatives
        # ---------------------------------------------------------------------
        # Bottom-biased derivative (D^-_y)
        Dy_m = self._hj_weno5_1d(
            v1=d_y[3:-3, 0:ny],   # delta_{j-5/2}
            v2=d_y[3:-3, 1:ny+1], # delta_{j-3/2}
            v3=d_y[3:-3, 2:ny+2], # delta_{j-1/2}
            v4=d_y[3:-3, 3:ny+3], # delta_{j+1/2}
            v5=d_y[3:-3, 4:ny+4]  # delta_{j+3/2}
        )
        
        # Top-biased derivative (D^+_y) using flipped inputs (v_tilde)
        Dy_p = self._hj_weno5_1d(
            v1=d_y[3:-3, 5:ny+5], # delta_{j+5/2}
            v2=d_y[3:-3, 4:ny+4], # delta_{j+3/2}
            v3=d_y[3:-3, 3:ny+3], # delta_{j+1/2}
            v4=d_y[3:-3, 2:ny+2], # delta_{j-1/2}
            v5=d_y[3:-3, 1:ny+1]  # delta_{j-3/2}
        )

        return Dx_m, Dx_p, Dy_m, Dy_p

    def _godunov_grad_norm(self, Dx_m: np.ndarray, Dx_p: np.ndarray, Dy_m: np.ndarray, Dy_p: np.ndarray, S0: np.ndarray) -> np.ndarray:
        """
        Computes the gradient norm using the classic Rouy-Tourin Godunov formulation.
        """
        # For S0 >= 0: max(D^-, -D^+, 0)^2 for each direction
        grad_plus = np.sqrt(
            np.maximum(np.maximum(Dx_m, -Dx_p), 0.0)**2 +
            np.maximum(np.maximum(Dy_m, -Dy_p), 0.0)**2
        )
        
        # For S0 < 0: max(-D^-, D^+, 0)^2 for each direction
        grad_minus = np.sqrt(
            np.maximum(np.maximum(-Dx_m, Dx_p), 0.0)**2 +
            np.maximum(np.maximum(-Dy_m, Dy_p), 0.0)**2
        )
        
        return np.where(S0 >= 0, grad_plus, grad_minus)

    def _compute_rhs(self, phi: np.ndarray, S0: np.ndarray, h: float) -> np.ndarray:
        """Computes the semi-discrete right-hand side: -S0 * (|grad_phi|_G - 1)."""
        Dx_m, Dx_p, Dy_m, Dy_p = self._get_derivatives_weno5(phi, h)
        grad_G = self._godunov_grad_norm(Dx_m, Dx_p, Dy_m, Dy_p, S0)
        return -S0 * (grad_G - 1.0)

    def reinitialize(self, phi0: np.ndarray, h: float, n_steps: int) -> np.ndarray:
        """Executes the SSP-RK3 pseudo-time integration."""
        if n_steps <= 0:
            return phi0.copy()

        # Enforce float64 for PDE iterations to avoid truncation errors
        phi = phi0.astype(np.float64, copy=True)
        S0 = self._smoothed_sign(phi, h)
        dt = self.cfl * h

        for _ in range(n_steps):
            L1 = self._compute_rhs(phi, S0, h)
            phi_1 = phi + dt * L1
            
            L2 = self._compute_rhs(phi_1, S0, h)
            phi_2 = 0.75 * phi + 0.25 * (phi_1 + dt * L2)
            
            L3 = self._compute_rhs(phi_2, S0, h)
            phi = (1.0/3.0) * phi + (2.0/3.0) * (phi_2 + dt * L3)

        return phi


class ReinitFieldPackBuilder:
    """
    Pipeline integrator: handles SDF pass-through and non-SDF batch reinitialization.
    """
    def __init__(self, cfl: float = 0.5):
        self.reinitializer = LevelSetReinitializer(cfl=cfl)
    
    def build(self, field_pack: Dict[str, Any], steps_list: List[int] = [5, 10, 15, 20]) -> Dict[str, Dict]:
        """
        Routes the field_pack based on whether it is an exact SDF or non-SDF.
        """
        # Defensive programming to prevent processing fields that were already reinitialized
        if field_pack["meta"].get("reinit") is not None:
            print(f"[Warning] Blueprint {field_pack['meta'].get('blueprint_id', 'Unknown')} already has reinit records. Skipping.")
            return {}

        phi0 = field_pack["field"]["phi"]
        h = field_pack["params"]["h"]
        
        # Determine if this is an SDF based on phi_type field
        phi_type = field_pack["field"].get("phi_type", "")
        is_sdf = "sdf" in phi_type and "nonsdf" not in phi_type
        
        # Route 1: SDF Pass-Through
        if is_sdf:
            pack_out = copy.deepcopy(field_pack)
            pack_out["field"]["phi"] = pack_out["field"]["phi"].astype(np.float32)
            return {"0": pack_out}

        # Route 2: Non-SDF Reinitialization Generation
        results = {}
        for steps in steps_list:
            phi_re = self.reinitializer.reinitialize(phi0, h, steps)
            metrics = ReinitQualityEvaluator.evaluate(phi_re, h)
            
            new_pack = copy.deepcopy(field_pack)
            new_pack["field"]["phi"] = phi_re.astype(np.float32) 
            new_pack["field"]["phi_type"] += f"_reinit{steps}"
            
            new_pack["meta"]["reinit"] = {
                "steps": steps,
                "scheme": "HJ-WENO5 + SSP-RK3 + Godunov (Rouy-Tourin)",
                "metrics_near_interface": metrics
            }
            results[str(steps)] = new_pack
            
        return results


In [4]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Dict, Any, List, Tuple
import copy

class HDF5DatasetCompiler:
    """
    Stage 4: Extracts 3x3 stencils from field_packs and writes them into an HDF5 database.
    Strictly follows Equation (13) formatting and grouping strategies.
    """
    def __init__(self, h5_filepath: str, mode: str = "w"):
        self.filepath = h5_filepath
        self.file = h5py.File(self.filepath, mode)

        if mode == "w":
            self.X_dset = self.file.create_dataset("X", shape=(0, 9), maxshape=(None, 9), dtype=np.float32, compression="gzip")
            self.Y_dset = self.file.create_dataset("Y", shape=(0, 1), maxshape=(None, 1), dtype=np.float32, compression="gzip")
            self.Steps_dset = self.file.create_dataset("reinit_steps", shape=(0, 1), maxshape=(None, 1), dtype=np.int32, compression="gzip")
            self.Blueprint_dset = self.file.create_dataset("blueprint_idx", shape=(0, 1), maxshape=(None, 1), dtype=np.int32, compression="gzip")
            self.Radius_dset = self.file.create_dataset("radius_idx", shape=(0, 1), maxshape=(None, 1), dtype=np.int32, compression="gzip")

  
    @staticmethod
    def extract_stencils(field_pack: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray]:
        phi = field_pack["field"]["phi"]          
        h = float(field_pack["params"]["h"])
        h_kappa = float(field_pack["label"]["h_kappa"])

        I, J = ReinitQualityEvaluator.get_sampling_coordinates(phi)
        
        if len(I) == 0:
            return np.zeros((0, 9), dtype=np.float32), np.zeros((0, 1), dtype=np.float32)

        patches = []
        for i, j in zip(I, J):
            # [CRITICAL FIX] Avoid Python's negative index boundary trap.
            # Slice normally first, then reverse columns and transpose.
            patch_2d = phi[i-1:i+2, j-1:j+2]
            patch_correct = patch_2d[:, ::-1].T
            patches.append(patch_correct.flatten())  

        patches = np.stack(patches, axis=0)    
        
        X_b =  patches.astype(np.float32)
        Y_b = np.full((X_b.shape[0], 1), h_kappa, dtype=np.float32)
        
        X_batch_aug = np.vstack([X_b, -X_b])
        Y_batch_aug = np.vstack([Y_b, -Y_b])
        
        return X_batch_aug, Y_batch_aug

    def append_data(self, field_packs: List[Dict[str, Any]]):
        """Appends batches and writes to HDF5. (Augmentation is already handled in extract_stencils)"""
        X_list, Y_list = [], []
        steps_list, bp_list, rad_list = [], [], []

        for pack in field_packs:
            # 这里的 X_b, Y_b 已经自带了 (-X, -Y) 的翻转增强
            X_b, Y_b = self.extract_stencils(pack)
            if X_b.shape[0] == 0:
                continue

            meta = pack["meta"]
            phi_type = pack["field"].get("phi_type", "")
            
            is_sdf = meta.get("is_sdf")
            if is_sdf is None:
                is_sdf = "sdf" in phi_type and "nonsdf" not in phi_type
            else:
                is_sdf = bool(is_sdf)

            if is_sdf:
                steps = 0
            else:
                reinfo = meta.get("reinit", None)
                if reinfo is None or "steps" not in reinfo:
                    raise KeyError(f"[Error] Non-SDF pack (phi_type: {phi_type}) missing meta['reinit']['steps'].")
                steps = int(reinfo["steps"])

            blueprint_idx = int(meta.get("blueprint_idx", -1))
            radius_idx = int(meta.get("radius_idx", -1))

            n_b = X_b.shape[0]
            
            X_list.append(X_b)
            Y_list.append(Y_b)
            steps_list.append(np.full((n_b, 1), steps, dtype=np.int32))
            bp_list.append(np.full((n_b, 1), blueprint_idx, dtype=np.int32))
            rad_list.append(np.full((n_b, 1), radius_idx, dtype=np.int32))

        if not X_list:
            return

        X_all = np.vstack(X_list)
        Y_all = np.vstack(Y_list)
        Steps_all = np.vstack(steps_list)
        BP_all = np.vstack(bp_list)
        Rad_all = np.vstack(rad_list)

        curr_len = self.X_dset.shape[0]
        new_len = curr_len + X_all.shape[0]

        self.X_dset.resize((new_len, 9))
        self.Y_dset.resize((new_len, 1))
        self.Steps_dset.resize((new_len, 1))
        self.Blueprint_dset.resize((new_len, 1))
        self.Radius_dset.resize((new_len, 1))

        self.X_dset[curr_len:new_len] = X_all
        self.Y_dset[curr_len:new_len] = Y_all
        self.Steps_dset[curr_len:new_len] = Steps_all
        self.Blueprint_dset[curr_len:new_len] = BP_all
        self.Radius_dset[curr_len:new_len] = Rad_all
    def close(self):
        self.file.close()

class LevelSetCurvatureDataset(Dataset):
    """
    PyTorch Dataset reading from the generated HDF5 file.
    Includes multiprocessing safety protocols for DataLoader compatibility.
    """
    def __init__(self, h5_filepath: str):
        super().__init__()
        self.filepath = h5_filepath
        self.file = None
        
        # Open briefly just to get the total length, then safely close
        with h5py.File(self.filepath, "r") as f:
            self.total_samples = f["X"].shape[0]

    def __len__(self) -> int:
        return self.total_samples

    def __getstate__(self):
        """
        [CRITICAL FIX] 
        This prevents PyTorch's DataLoader from crashing when num_workers > 0.
        It drops the un-picklable HDF5 file handle before distributing the dataset 
        to background worker processes.
        """
        state = self.__dict__.copy()
        state["file"] = None  
        return state

    def _ensure_open(self):
        """Lazy loading ensures every worker process opens its own safe file handle."""
        if self.file is None:
            self.file = h5py.File(self.filepath, "r")
            
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int, int, int]:
            self._ensure_open()
            
            x = np.array(self.file["X"][idx])            
            y = np.array(self.file["Y"][idx])            
            
            # Wrapped in np.array() to satisfy the VS Code / Pylance type checker
            step = int(np.array(self.file["reinit_steps"][idx])[0])
            bp_idx = int(np.array(self.file["blueprint_idx"][idx])[0])
            rad_idx = int(np.array(self.file["radius_idx"][idx])[0])

            return torch.from_numpy(x), torch.from_numpy(y), step, bp_idx, rad_idx
    def close(self):
        if self.file is not None:
            self.file.close()
            self.file = None

In [5]:
import os
import h5py
import numpy as np
from tqdm import tqdm

def generate_full_dataset_for_resolution(rho: int, h5_filename: str):
    """
    End-to-end generation of complete multi-million sample dataset for a given resolution.
    Includes safe file handling and rigorous error catching.
    """
    if os.path.exists(h5_filename):
        os.remove(h5_filename)
        
    generator = CircleGeometryGenerator(resolution_rho=rho, seed=42, variations=5)
    field_builder = LevelSetFieldBuilder(dtype=np.float64)
    reinit_builder = ReinitFieldPackBuilder(cfl=0.5)
    
    print(f"\n>>> Starting Pipeline for Resolution: {rho}")
    dataset_compiler = HDF5DatasetCompiler(h5_filename, mode='w')

    try:
        blueprints = generator.generate_blueprints()
        print(f"[*] Successfully generated {len(blueprints)} circle blueprints.")

        # TQDM progress bar with dynamic description
        for bp in tqdm(blueprints, desc=f"Processing rho={rho}"):
            
            pack_sdf = field_builder.build_circle_sdf(bp, return_grid=False)
            pack_nonsdf = field_builder.build_circle_nonsdf(bp, return_grid=False)
            
            out_sdf = reinit_builder.build(pack_sdf)
            out_nonsdf = reinit_builder.build(pack_nonsdf, steps_list=[5, 10, 15, 20])
            
            # Collect the 5 generated fields into a list
            batch_packs = [out_sdf["0"]]
            for step in ["5", "10", "15", "20"]:
                batch_packs.append(out_nonsdf[step])
                
            # Extract 3x3 stencils, perform augmentation, and append to HDF5
            dataset_compiler.append_data(batch_packs)
            
        print(f"[*] Dataset generation complete! Saved to {h5_filename}")
        
    except Exception as e:
        print(f"\n[!] Pipeline interrupted for rho={rho} due to error: {e}")
        raise
    finally:
        dataset_compiler.close()

if __name__ == "__main__":
    # Target resolutions specified in the paper
    target_rhos = [256, 266, 276]
    output_dir = "data"
    os.makedirs(output_dir, exist_ok=True)
    
    print("Launching Ultimate Data Generation Pipeline")
    
    for rho in target_rhos:
        db_name = os.path.join(output_dir, f"train_rho{rho}.h5")
        generate_full_dataset_for_resolution(rho, db_name)
    
    print(" Final Verification Results (Check against Paper Table 1)")

    for rho in target_rhos:
        db_name = os.path.join(output_dir, f"train_rho{rho}.h5")
        if os.path.exists(db_name):
            with h5py.File(db_name, 'r') as f:
                total_samples = f['X'].shape[0]
                print(f"Resolution {rho}: {total_samples:>9,} samples generated -> {db_name}")
        else:
            print(f"Resolution {rho}: Failed to locate {db_name}")

C:\Users\29801\AppData\Local\Temp\ipykernel_74320\3819931530.py:239: RuntimeWarning: overflow encountered in scalar multiply
  x ^= np.uint64(v_idx + 1) * np.uint64(14029467366897019727)


Launching Ultimate Data Generation Pipeline

>>> Starting Pipeline for Resolution: 256
[*] Successfully generated 620 circle blueprints.


Processing rho=256: 100%|██████████| 620/620 [31:21<00:00,  3.03s/it]


[*] Dataset generation complete! Saved to data\train_rho256.h5

>>> Starting Pipeline for Resolution: 266
[*] Successfully generated 645 circle blueprints.


Processing rho=266: 100%|██████████| 645/645 [26:58<00:00,  2.51s/it]


[*] Dataset generation complete! Saved to data\train_rho266.h5

>>> Starting Pipeline for Resolution: 276
[*] Successfully generated 670 circle blueprints.


Processing rho=276: 100%|██████████| 670/670 [21:51<00:00,  1.96s/it]

[*] Dataset generation complete! Saved to data\train_rho276.h5
 Final Verification Results (Check against Paper Table 1)
Resolution 256: 3,151,720 samples generated -> data\train_rho256.h5
Resolution 266: 3,408,020 samples generated -> data\train_rho266.h5
Resolution 276: 3,673,960 samples generated -> data\train_rho276.h5


In [6]:
import h5py
import numpy as np

def validate_curvature_dataset(h5_filepath: str, rho: int = 256):
    print(f"Starting comprehensive dataset validation: {h5_filepath} (rho={rho})")

    h = 1.0 / (rho - 1)
    
    try:
        with h5py.File(h5_filepath, 'r') as f:
            X = f['X'][:]
            Y = f['Y'][:]
            steps = f['reinit_steps'][:]
            
            total_samples = X.shape[0]
            print(f" [1/5] Basic dimension check: Successfully loaded {total_samples:,} samples.")
            assert X.shape[1] == 9, "Feature dimension must be 9 (3x3 template)"
            assert Y.shape[1] == 1, "Label dimension must be 1"
            
            # --- Check 2: NaN and Inf anomaly detection ---
            print(" [2/5] Data purity check: Searching for NaN or Inf anomalies...")
            assert not np.isnan(X).any(), "Fatal error: X contains NaN!"
            assert not np.isnan(Y).any(), "Fatal error: Y contains NaN!"
            assert not np.isinf(X).any(), "Fatal error: X contains Inf!"
            
            # --- Check 3: Symmetry and data augmentation verification (Symmetry) ---
            print(" [3/5] Data augmentation symmetry check...")
            # Because we strictly add paired (X, Y) and (-X, -Y) during extraction
            # The global mean should approach 0
            x_mean = np.abs(np.mean(X))
            y_mean = np.abs(np.mean(Y))
            assert x_mean < 1e-6, f"Symmetry failed: X mean {x_mean} does not approach 0"
            assert y_mean < 1e-6, f"Symmetry failed: Y mean {y_mean} does not approach 0"
            
            # --- Check 4: Label theoretical bound verification (Label Bounds) ---
            print(" [4/5] Curvature label theoretical bound check...")
            r_min = 1.6 * h
            r_max = 0.5 - 2.0 * h
            # Theoretical maximum and minimum dimensionless curvature (both positive and negative)
            hk_max = float(h / r_min)  # ~0.625
            hk_min = float(h / r_max)  # ~0.008
            
            y_abs = np.abs(Y)
            max_y_val = np.max(y_abs)
            min_y_val = np.min(y_abs)
            
            print(f"    - Theoretical h*kappa range: [{hk_min:.4f}, {hk_max:.4f}]")
            print(f"    - Actual h*kappa range: [{min_y_val:.4f}, {max_y_val:.4f}]")
            # Allow small floating-point error
            assert max_y_val <= hk_max * 1.01, f"🚨 Label anomaly: Maximum curvature exceeded ({max_y_val})"
            assert min_y_val >= hk_min * 0.99, f"🚨 Label anomaly: Minimum curvature undershot ({min_y_val})"

            # --- Check 5: Eikonal equation gradient norm verification (|∇φ| ≈ 1) ---
            print(" [5/5] Eikonal physical law check (|∇φ| ≈ 1)...")
            # Only check pure native SDF samples strictly (non-reinitialized may have tiny deviations)
            sdf_mask = (steps[:, 0] == 0)
            X_sdf = X[sdf_mask]
            
            if len(X_sdf) > 0:
                # Randomly select 1000 native SDF samples for central difference
                idx = np.random.choice(len(X_sdf), min(1000, len(X_sdf)), replace=False)
                sample_X = X_sdf[idx]
                
                # Recover gradients based on Eq.(13) specific arrangement
                # Expansion order: [0:i-1,j+1], [1:i,j+1], [2:i+1,j+1]
                #                 [3:i-1,j],   [4:i,j],   [5:i+1,j]
                #                 [6:i-1,j-1], [7:i,j-1], [8:i+1,j-1]
                # Note X is already h*phi, so true phi = X / h
                # phi_x = (phi[i+1,j] - phi[i-1,j]) / 2h = (X[5]/h - X[3]/h) / 2h = (X[5] - X[3]) / (2 * h^2)
                # phi_y = (phi[i,j+1] - phi[i,j-1]) / 2h = (X[1]/h - X[7]/h) / 2h = (X[1] - X[7]) / (2 * h^2)
                
                phi_x = (sample_X[:, 5] - sample_X[:, 3]) / (2.0 * h**2)
                phi_y = (sample_X[:, 1] - sample_X[:, 7]) / (2.0 * h**2)
                
                grad_norm = np.sqrt(phi_x**2 + phi_y**2)
                mean_grad = np.mean(grad_norm)
                
                print(f"    - SDF sample average gradient norm: {mean_grad:.6f} (theoretical value: 1.000000)")
                # Central difference on circular discrete grids typically in range [0.95, 1.05]
                assert 0.90 < mean_grad < 1.10, "Physical law violation: Interface gradient norm severely deviates from 1!"
            

    except Exception as e:
        print(f"\nValidation failed! Data defect detected: {e}")

if __name__ == "__main__":
    
    validate_curvature_dataset("data/train_rho256.h5", rho=256)

Starting comprehensive dataset validation: data/train_rho256.h5 (rho=256)
 [1/5] Basic dimension check: Successfully loaded 3,151,720 samples.
 [2/5] Data purity check: Searching for NaN or Inf anomalies...
 [3/5] Data augmentation symmetry check...
 [4/5] Curvature label theoretical bound check...
    - Theoretical h*kappa range: [0.0080, 0.6250]
    - Actual h*kappa range: [0.0080, 0.6250]
 [5/5] Eikonal physical law check (|∇φ| ≈ 1)...
    - SDF sample average gradient norm: 254.964661 (theoretical value: 1.000000)

Validation failed! Data defect detected: Physical law violation: Interface gradient norm severely deviates from 1!
